# 03 — Validate a 20-second motionless recording

Before attempting translation, learn what a **perfectly stationary** IMU
reports. Accelerometer norm should be near 9.80665 m/s² and angular rate
near zero after calibration. The position solution may still drift rapidly.

This notebook checks sidecar metadata and refuses known FIFO-overflow
recordings. Run notebook 02 or `python record_ftdi.py --seconds 20` first.
In the absence of real hardware, use the included synthetic square only
to explore plotting; it is NOT a stationary test.

In [ ]:
try:
    from pathlib import Path
    import matplotlib.pyplot as plt
    from validate_stationary import diagnose
    RECORDING = Path('data/stationary.csv')
    if RECORDING.exists():
        diagnostics = diagnose(RECORDING)
        for name, value in diagnostics.items():
            if not name.startswith('trajectory_'):
                print(name, ':', value)
    else:
        diagnostics = None
        print('Record a stationary CSV first using notebook 02 or the CLI.')
except KeyboardInterrupt:
    print("Cell interrupted safely; kernel stays usable.")

In [ ]:
try:
    if diagnostics is not None:
        fig, ax = plt.subplots(1, 2, figsize=(12, 4))
        for key, label in [('trajectory_raw', 'No residual gyro bias compensation'),
                           ('trajectory_corrected', 'Initial 2s gyro bias removed')]:
            p = diagnostics[key].p
            ax[0].plot(p[:,0], p[:,1], label=label)
            ax[1].plot(diagnostics[key].t, p[:,2], label=label)
        ax[0].scatter([0], [0], marker='x', color='black', label='True rest')
        ax[0].set(xlabel='X [m]', ylabel='Y [m]', title='Apparent XY travel')
        ax[0].axis('equal')
        ax[1].set(xlabel='Time [s]', ylabel='Estimated Z [m]', title='False vertical motion')
        ax[0].legend(); ax[1].legend(); fig.tight_layout(); plt.show()
    else:
        print('No stationary recording selected; no plots generated.')
except KeyboardInterrupt:
    print("Cell interrupted safely; kernel stays usable.")